# Seminar HCI and BCI in practice
## Session 7 Evaluation

***In this session the performance of the SVM classifier will be evaluated.***


In [ ]:
import numpy as np
import os
import sys
import pickle
from scipy import stats
import matplotlib.pyplot as plt

sys.path.append(os.path.join(os.getcwd(), "src"))
from nearly import nearly
from classification_svm_session07 import classification_svm
from gen_selector import gen_selector
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data')
print(f'Now you are located: {main_path}')


In [ ]:
ecog_file = os.path.join(data_path, 'raw/ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

# Load epoch info
epoch_file = os.path.join(data_path, 'raw/epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print("\nepoch info:")
for key, value in epoch.items():
    print(f"Key:{key}, Type:{type(value)}")

## 1. Define the dataset for the classification based on previously found features (use your results from Session 5)

In [ ]:
# Frequency features (4-58, 62-118, 122-178 Hz based on Session 5 results)
# You can change the values here, based on your results from Session 5 (t-value plot)
freqBand = np.concatenate([np.arange(62, 119), np.arange(122, 179)])  # just for faster testing

# Find nearest frequency indices
freqIdx = np.unique(nearly(freqBand, ecog['periodogram']['centerFrequency']))
# Alternative:
# freqIdx = np.unique([np.argmin(np.abs(ecog['periodogram']['centerFrequency'] - f)) for f in freqBand])
nFreq = len(freqIdx)

# Number of trials with finger movement
nTrials = np.array(ecog['periodogram']['periodogram']).shape[2]

# Channel features (based on Session 5 results)
chan = np.array([17, 23, 39]) - 1  # Convert to 0-based indexing
nChan = len(chan)

# Prepare data for z-scoring (same as Session 4)
# Reshape to (nFreq, nChan*nTrials)
dat = np.array(ecog['periodogram']['periodogram'])[freqIdx, :, :][:, chan, :]
dat = dat.reshape(nFreq, nChan * nTrials, order='F')

# Z-score data along frequency axis
dat = stats.zscore(dat, axis=1) 

# Reshape data back to original structure with permutations
dat = dat.reshape(nFreq, nChan, nTrials, order='F')
dat = np.transpose(dat, (2, 1, 0)) 
dat = dat.reshape(nTrials, nFreq * nChan, order='F')

# Get the true labels for data
realClassLabels = np.array(epoch['label'])

# Take a look into the data shape now, and understand what are the dimensions of data
print("Understand the dimensions of data by yourself")
print(f"dat shape now is: {dat.shape}")
print(f"real labels for data now is: {realClassLabels.shape}")
print(f"Number of 21 class labels:{np.count_nonzero(realClassLabels == 21)}")
print(f"Number of 20 class labels:{np.count_nonzero(realClassLabels == 20)}")

---

## 2 Classification error

### 2.1 Cross Validation

In [ ]:
CV_steps = 10  # CV-steps
rep_nr = 3  # performing the same classification x times (you can change this value)

svm_results_all = []

for i in range(rep_nr):
    print(f'CV-Repeat #{i+1}')
    selector = gen_selector(len(realClassLabels), CV_steps, random_seed=i)

    # SVM Model in repetition #i
    svm_results = classification_svm(dat, realClassLabels, selector, CV_steps, optimizeC=True, dispC=False)
 
    # Save SVM Results into a list, each element of the list is a dict, which contains svm results in single rep
    svm_results_all.append(svm_results)

for key, value in svm_results_all[0].items():
    print(f"Key:{key}, Type:{type(value)}")


<h2 style="color: #FF0000; font-weight: bold;">TASK 1 Discussion (2 pt):</h2>

- Explain the outputs: What information can you get from dict `svm_results`? (2 pt)

In [ ]:
# --- TASK 1: what is inside svm_results? ---
res = svm_results_all[0]      # look at the first repetition

print("svm_results has %d keys:\n" % len(res))
for key, value in res.items():
    if key == 'C_plus_W':
        print("  %-22s list of %d tuples (C, norm of w)" % (key, len(value)))
    elif np.ndim(value) == 0:
        print("  %-22s single number = %.4f" % (key, value))
    else:
        print("  %-22s shape %s" % (key, np.array(value).shape))

print("\noverall accuracy      : %.2f%%" % (res['accuracy'] * 100))
print("accuracy of each fold : ", np.round(res['accuracies'], 3))
print("mean of the folds     : %.4f  (same as the overall accuracy)" % res['accuracies'].mean())
print("C chosen in each fold : ", np.round(res['best_Cs'], 5))

# what do the decision values mean?
dv = res['decision_values']
pred = res['predictedClassLabels']
print("\ndecision_values go from %.2f to %.2f" % (dv.min(), dv.max()))
print("where the value is negative, the prediction is:", np.unique(pred[dv < 0]))
print("where the value is positive, the prediction is:", np.unique(pred[dv > 0]))
print("-> the sign gives the class, the size says how sure the SVM is")

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

`svm_results` is a dict with **8 keys**. The classification is done with 10-fold cross-validation, so some entries are one value per fold and some are one value per trial.

| key | shape | what it contains |
| :--- | :--- | :--- |
| `accuracy` | single number | the accuracy over **all** trials together, here **75.48%** |
| `accuracies` | (10,) | the accuracy of **each single fold**, here between 67.7% and 87.1% |
| `best_Cs` | (10,) | the C value that was chosen in each fold |
| `weights` | (10, 177) | the weight vector of each fold, one weight per feature |
| `biases` | (10,) | the bias (intercept) of the SVM in each fold |
| `predictedClassLabels` | (314,) | the predicted label (20 or 21) for every trial |
| `decision_values` | (314,) | how far every trial is from the decision boundary |
| `C_plus_W` | list of 10 | pairs of (C, length of the weight vector) |

**What I can learn from it:**

- **`accuracy` and `accuracies`.** The single accuracy tells me how good the classifier is in total, and `accuracies` shows how much it changes between the folds. Here it goes from 67.7% to 87.1%, so almost 20 percentage points difference. That is a lot and it shows that with only about 31 test trials per fold, a single fold result is not very reliable. This is exactly why the whole thing is repeated 3 times in the cell above and the mean plus the standard error is plotted. The mean of the 10 folds is the same as the overall accuracy, because the folds have almost the same size.

- **`best_Cs`.** The C that the optimisation picked in each fold. The values jump around a lot (from 0.0001 to 0.31), which is plotted as a histogram in the next cell. As I already found in Session 6, C is selected on the test set here, so these values are not very stable.

- **`weights`.** This is the interesting part for interpretation. Each row is one fold, and each column is one of the 177 features (59 frequencies x 3 channels). A large weight means that this feature was important for the decision. Because there are 10 folds, we get 10 weight vectors, and in section 2.2 they are used to look at the best one and at the average one. If a feature has a large weight in **all** folds, it is really useful, and if it is large in only one fold, it was probably noise.

- **`biases`.** The offset of the decision boundary. Together with the weights it defines the line that separates the two classes.

- **`predictedClassLabels`.** The guessed label for every trial, 20 for flexion and 21 for extension. Every trial got its prediction from the fold in which it was in the test set, so no trial was predicted by a classifier that had seen it before.

- **`decision_values`.** This one is new in this session and is needed for the ROC curve later. Instead of only saying "class 20" or "class 21", it says how far away from the boundary the trial is. I checked it in the cell: where the value is **negative** the prediction is 20, where it is **positive** it is 21. So the **sign gives the class** and the **size tells how sure** the SVM is. A trial with a value of -10 is a clear case, a trial with -0.1 is almost on the border. The ROC curve uses this to move the threshold instead of always cutting at 0.

- **`C_plus_W`.** For every fold the C that was used together with the length of the weight vector. This shows the effect of C directly: a small C gives small weights and a simpler model.

In [ ]:
# Plot mean and standard error of accuracy across X repetitive classifications

accuracy_svm = [result['accuracy'] for result in svm_results_all]
mean_acc = np.mean(accuracy_svm)
std_error = np.std(accuracy_svm) / np.sqrt(len(accuracy_svm))

plt.figure(figsize=(8, 6))
plt.errorbar(0, mean_acc, yerr=std_error, 
            fmt='x', color='red', 
            markersize=10, capsize=5,
            label='SVM Accuracy')

plt.xlim(-0.5, 0.5)
plt.ylim(mean_acc-2*std_error, mean_acc+2*std_error)  
plt.xticks([]) 
plt.ylabel('Classification Accuracy')
plt.title('SVM Classification Performance')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)


plt.text(0.1, mean_acc+0.2*std_error, 
         f'Mean: {mean_acc:.3f}\nSE: {std_error:.3f}',
         ha='left', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
best_Cs_all = np.array([result['best_Cs'] for result in svm_results_all])
best_Cs_1d = best_Cs_all.ravel()

plt.figure(figsize=(10, 6))  
n, bins, patches = plt.hist(best_Cs_1d, 
                           bins='auto',  
                           color='skyblue',
                           edgecolor='black',
                           alpha=0.7)
plt.xlabel('C Values', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of C Values', fontsize=14, pad=20)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i in range(len(n)):
    if n[i] > 0:  
        plt.text(bins[i] + (bins[i+1]-bins[i])/2, n[i], 
                str(int(n[i])), 
                ha='center', 
                va='bottom')

plt.tight_layout() 
plt.show()

---

### 2.2 SVM weights: FIND BEST CROSS VALIDATION

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 Find best CV-Step Code (2 pt)</h2>

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration (finish the ... part in the following code cell): </h3>


In [ ]:
# # Choose the fold step and repetition containing the highest accuracy
# Hint 1: `accuracy_svm_all` is a list, in which the accuracy rates over all folds and repetitions are saved
# Hint 2: check function `argmax` in numpy (https://numpy.org/devdocs/reference/generated/numpy.argmax.html 27/05/25)
# Hint 3: check function `np.unravel_index` in numpy (https://numpy.org/doc/stable/reference/generated/numpy.unravel_index.html 12/06/25)
accuracies_svm_all = [result['accuracies'] for result in svm_results_all]
max_cv_accuracy = np.max(accuracies_svm_all)
best_cv_idx = np.argmax(accuracies_svm_all)

best_rep_idx, best_cv_idx = np.unravel_index(best_cv_idx, (rep_nr, CV_steps))

### 2.2.2 Plot the weights for best W VECTOR:


In [ ]:
# `weights_all` saves all weights from all repetitions and all CVs
weights_all = np.array([result['weights'] for result in svm_results_all])
# Select the weight-value for the best rep and best CV
best_w = weights_all[best_rep_idx, best_cv_idx,:]

reshaped_w = best_w.reshape(nFreq, nChan, order = 'c').T

plt.figure(figsize=(12, 8))
img = plt.imshow(abs(reshaped_w), aspect='auto', cmap='RdBu_r')

freqTicks = np.arange(freqBand[0], freqBand[1] + 1, 10)
tickPos = np.linspace(0, nFreq - 1, len(freqTicks), dtype=int)

plt.yticks(ticks=np.arange(nChan), labels=chan, fontsize=12)
plt.xticks(ticks=tickPos, labels=freqTicks, fontsize=12)
plt.xlabel('Frequency (Hz)', fontsize=14, fontweight='bold')
plt.ylabel('Electrode', fontsize=14, fontweight='bold')
plt.colorbar(img)

plt.tight_layout()
plt.show()

### 2.2.3 Plot AVERAGE w vector:

In [ ]:
# Average W vector
avW = np.mean(weights_all, axis = (0,1))
reshaped_avW = avW.reshape(nFreq, nChan, order = 'c').T

plt.figure(figsize=(12, 8))
img = plt.imshow(abs(reshaped_avW), aspect='auto', cmap='RdBu_r')

freqTicks = np.arange(freqBand[0], freqBand[1] + 1, 10)
tickPos = np.linspace(0, nFreq - 1, len(freqTicks), dtype=int)

plt.yticks(ticks=np.arange(nChan), labels=chan, fontsize=12)
plt.xticks(ticks=tickPos, labels=freqTicks, fontsize=12)
plt.xlabel('Frequency (Hz)', fontsize=14, fontweight='bold')
plt.ylabel('Electrode', fontsize=14, fontweight='bold')
plt.colorbar(img)

plt.tight_layout()
plt.show()


<h2 style="color: #FF0000; font-weight: bold;">TASK 3 Discussion (1 pt)</h2>

- Explain what you see in these plots.

In [ ]:
# --- TASK 3: compare the best weight vector with the average one ---
freqHz = np.array(ecog['periodogram']['centerFrequency'])[freqIdx]   # it is a list, so wrap it first

best_matrix = best_w.reshape(nFreq, nChan, order='c').T     # (nChan, nFreq)
avg_matrix  = avW.reshape(nFreq, nChan, order='c').T

print("best fold: repetition %d, CV step %d, accuracy %.1f%%"
      % (best_rep_idx, best_cv_idx, accuracies_svm_all[best_rep_idx][best_cv_idx] * 100))

print("\n%9s %14s %14s" % ("channel", "best |w| mean", "avg |w| mean"))
for k in range(nChan):
    print("%9d %14.4f %14.4f"
          % (chan[k] + 1, np.abs(best_matrix[k]).mean(), np.abs(avg_matrix[k]).mean()))

print("\nwhere is the strongest weight of each channel?")
for k in range(nChan):
    print("   ch %2d : best fold peaks at %3.0f Hz , average peaks at %3.0f Hz"
          % (chan[k] + 1,
             freqHz[np.argmax(np.abs(best_matrix[k]))],
             freqHz[np.argmax(np.abs(avg_matrix[k]))]))

print("\nlength of the weight vector: best = %.2f , average = %.2f"
      % (np.linalg.norm(best_w), np.linalg.norm(avW)))

# how similar are the weight vectors of the different folds?
flat = weights_all.reshape(rep_nr * CV_steps, -1)
cc = np.corrcoef(flat)
iu = np.triu_indices_from(cc, 1)
print("mean correlation between the %d weight vectors: %.2f" % (rep_nr * CV_steps, cc[iu].mean()))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

Both plots show the **weights of the SVM** as a channels x frequencies image, so the same layout as the t-value plot from Session 5. The y-axis are my 3 channels (17, 23, 39), the x-axis are the frequencies, and the colour is the size of the weight. A large weight means the SVM used that feature a lot to decide between flexion and extension, and a weight near 0 means the feature was almost ignored. The plots use `abs()`, because a strong negative weight is just as important as a strong positive one, only the direction is different.

**The first plot (best W vector)** is the weight vector of the single fold that had the highest accuracy, here repetition 0, CV step 1 with **87.1%**. It looks **very patchy**: there are a few very bright spots and a lot of dark area around them. The strongest values are around 0.72 to 0.75, while the average weight of a channel is only about 0.24, so single features stick out a lot.

**The second plot (average W vector)** is the mean over all 30 weight vectors (3 repetitions x 10 folds). It looks much **smoother and weaker**. The length of the weight vector drops from **3.93 to 0.88**, so the average is about 4 times smaller. That happens because the weights of the different folds do not point in exactly the same direction, so when they are averaged they partly cancel each other out. I checked this: the mean correlation between the 30 weight vectors is only **0.51**, so the folds agree only about half of the time.

**Which one should I trust?** The average one. The best fold only had about 31 test trials, so it can be high by luck, and part of its bright spots is just noise of that one split. The average uses all 30 folds, so what is left over is what the classifier found **again and again**, and the noise averages out.

**What stays the same in both plots** is the interesting part:

| channel | peak in the best fold | peak in the average |
| ---: | ---: | ---: |
| 17 | **157 Hz** | **157 Hz** |
| 23 | 129 Hz | 64 Hz |
| 39 | **105 Hz** | **105 Hz** |

Channel 17 at about 157 Hz and channel 39 at about 105 Hz appear in **both** plots, so these are real and stable features. Channel 23 is different in the two plots, which means its peak in the best fold was probably a coincidence of that split.

The nicest thing is that this fits with **Session 5**: there the t-value plot showed channel 17 as the strongest channel with |t| = 7.15 at exactly **157 Hz**. So a simple t-test and the SVM weights, which are two completely different methods, point at the same feature. That makes me quite confident that this really is where the information about the finger movement is, and it is in the high gamma range as expected.

The weights are also fairly spread out over the frequencies instead of sitting in one single line. That means the classifier does not use one narrow frequency but a whole band, which is why in Session 6 taking only every 10th frequency worked just as well.

## 2.3 Area under curve (ROC: receiver operation characteristics)

In [ ]:
from plot_ROC import plot_ROC


In [ ]:
auc, *_ = zip(*[plot_ROC(result, realClassLabels, do_plot=True) for result in svm_results_all])

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 Discussion (1 pt)</h2>

- questions in regards to plots created by the function plot_ROC above

1. What does the first plot (distances to the hyperplane) portray.
2. How would this plot ideally look like?
3. What information can you get from the second plot?
4. What is a ROC?
5. What does the Area under the ROC mean?
6. How would this plot ideally look like?

In [ ]:
# --- TASK 4: look at the numbers behind the two ROC plots ---
res = svm_results_all[0]
bestFold = np.argmax(res['accuracies'])
W = res['weights'][bestFold, :]
distances = res['decision_values'] / np.linalg.norm(W)   # same as in plot_ROC

d20 = distances[realClassLabels == 20]
d21 = distances[realClassLabels == 21]

print("distance to the hyperplane:")
print("   class 20 : mean %+.2f   sd %.2f   (%d trials)" % (d20.mean(), d20.std(), len(d20)))
print("   class 21 : mean %+.2f   sd %.2f   (%d trials)" % (d21.mean(), d21.std(), len(d21)))

print("\nhow much do the two histograms overlap?")
print("   %d of %d class-20 trials lie on the class-21 side" % ((d20 > 0).sum(), len(d20)))
print("   %d of %d class-21 trials lie on the class-20 side" % ((d21 < 0).sum(), len(d21)))
print("   -> %d wrong out of %d, so accuracy = %.1f%%"
      % ((d20 > 0).sum() + (d21 < 0).sum(), len(realClassLabels), res['accuracy'] * 100))

print("\nAUC of each repetition:", [round(a, 3) for a in auc])
print("mean AUC: %.3f" % np.mean(auc))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**1. What does the first plot (distances to the hyperplane) portray?**

The SVM separates the two classes with a hyperplane. For every trial we can compute how far it lies from that plane, and on which side. That is what `decision_values` contains, divided by the length of the weight vector so it becomes a real distance. The plot is a **histogram of these distances**, one colour for the flexion trials (class 20, blue) and one for the extension trials (class 21, red).

The sign says which side the trial is on, so it says which class the SVM predicted, and the size says how far away it is, so how sure the SVM is. A trial far from the plane is a clear case, a trial close to 0 is almost on the border and could easily go either way.

In my data the two histograms are shifted against each other, class 20 has a mean distance of **-0.19** and class 21 of **+0.11**, so they really do sit on different sides. But they also **overlap a lot** in the middle: **48 of the 163** class-20 trials end up on the class-21 side and **29 of the 151** class-21 trials on the class-20 side. Together that is 77 wrong trials out of 314, which is exactly the 75.5% accuracy.

**2. How would this plot ideally look like?**

Ideally the two histograms would be **completely separated**, with no overlap at all: all blue bars clearly on the left of 0 and all red bars clearly on the right, with an empty gap in the middle. That would mean every trial is on the correct side and the classifier makes no mistakes.

It would also be nice if both histograms were **far away from 0** and narrow, because then even the worst trial is still a clear case and small changes in the data would not flip the decision.

**3. What information can you get from the second plot?**

The second plot is the ROC curve, and it shows **how good the classifier is at every possible threshold**, not just at the standard cut at 0. The x-axis is the false positive rate (class 20 trials wrongly called 21) and the y-axis is the true positive rate (class 21 trials correctly found). The dashed diagonal line is what pure guessing would give.

The further the orange curve bends up into the top left corner, the better. It also tells me what I can trade: if I want to catch more extension trials, I have to accept more false alarms, and the curve shows exactly how much.

**4. What is a ROC?**

ROC means Receiver Operating Characteristic. Instead of always deciding at distance 0, it moves the threshold through **all** possible values. For every threshold it computes the true positive rate and the false positive rate, and plots them against each other. So the ROC is the complete picture of the classifier over all thresholds, while the accuracy is only one single point on it.

The advantage is that the ROC does not depend on the threshold and it is not fooled by unequal class sizes, which is why it is often more useful than the accuracy alone.

**5. What does the area under the ROC mean?**

The AUC is the area under that curve, so a number between 0 and 1. It has a nice meaning: if I take one random class-20 trial and one random class-21 trial, **the AUC is the probability that the classifier gives the class-21 trial the higher decision value**. So it is the chance that the classifier puts the two in the right order.

- AUC = 0.5 means guessing, the curve is the diagonal.
- AUC = 1.0 means perfect separation.
- AUC below 0.5 means it is worse than guessing, which usually means the classes are swapped.

In my runs the AUC is **0.780, 0.764 and 0.790**, so on average **0.78**. That means in about 78 out of 100 random pairs the classifier ranks the two trials correctly. It is clearly better than the 0.5 of guessing, but far from perfect, which fits with the overlap I saw in the first plot.

**6. How would this plot ideally look like?**

Ideally the curve would go **straight up from (0,0) to (0,1) and then straight right to (1,1)**, so it would follow the top left corner exactly. That would mean there is a threshold at which the classifier finds all class-21 trials without a single false alarm, and the AUC would be 1.0.

My curve is clearly above the diagonal, so the classifier learned something real, but it is a smooth bend and not a corner, which is the same information as the overlapping histograms in the first plot.

## 2.4 Estimation of the chance level: Estimating Chance Level & CV

You have already tuned the `best C`, now using the `best repetition` and `best CV-step` to index the `best C-value`

In [ ]:
# Now we want to bootstrap the data
# Redefine some variables, in case you have already forgot the parameters
CV_step = 10  # CV-steps
rep_nr = 400  # performing the same classification x times (now is the bootstrapping times)
realClassLabels = np.array(epoch['label'])
accuracy_permuted = np.zeros(rep_nr)
best_C = best_Cs_all[best_rep_idx, best_cv_idx]

for i in range(rep_nr):
    # print(f'CV-Repeat #{i+1}')
    # randomly generate CV-fold series for the data
    selector = gen_selector(len(realClassLabels), CV_step, random_seed=i)  #using index as random_seed for reprocduce of data results

    # SVM Model in repetition #i
    for k in range(1, CV_step + 1):
        
        # Split data into train/test sets
        testIdx = np.where(selector == k)[0]
        trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)

        X_train = dat[trainIdx, :]
        y_train = realClassLabels[trainIdx]
        X_test = dat[testIdx, :]
        y_test = realClassLabels[testIdx]

        #!!! Key step of this permutation test
        # Permute (shuffle) y_train to break the true relationship with X_train
        y_train_permuted = np.random.permutation(y_train)  # Randomly shuffles labels

        # Random label test
        svm_permuted = SVC(kernel='linear', C=best_C)
        svm_permuted.fit(X_train, y_train_permuted)  # Train on shuffled labels
    score = svm_permuted.score(X_test, y_test)
    accuracy_permuted[i] = score


In [ ]:
# plot multiple errorbars
methods = ['SVM', 'SVM permutation_test']
means = [np.mean(accuracy_svm), np.mean(accuracy_permuted)]
stds = [np.std(accuracy_svm), np.std(accuracy_permuted)]
colors = ['#1f77b4', '#ff7f0e']
text_pos = [0.12, 1.03]

plt.figure(figsize=(8, 6))
for i in range(len(methods)):
    plt.errorbar(methods[i], means[i], yerr=1.96*np.array(stds[i]),
                 fmt='x', markersize=10, capsize=5, capthick=2, color=colors[i])
plt.ylim(min(means[0]-3*stds[0], means[1]-3*stds[1]), 
         max(means[0]+3*stds[0], means[1]+3*stds[1]))
plt.xlim(-0.2, 1.65)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison (95% CI)')
for i in range(len(methods)):
    plt.text(text_pos[i], means[i]-0*stds[i], 
             f'{methods[i]}:\n {means[i]:.3f} ± {1.96*stds[i]:.3f}',
             ha='left', va='center', color=colors[i])

plt.grid(True, alpha=0.3)
plt.show()

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 Discussion (2 pt)</h2>

- How is the chance level estimated and for what do you use it here?
- Why is this an estimation of the chance level?

In [ ]:
# --- TASK 5: the permutation test, and how far the real result is from chance ---
real_acc = np.mean(accuracy_svm)

print("real SVM accuracy            : %.1f%%" % (real_acc * 100))
print("permutation test, mean       : %.1f%%" % (accuracy_permuted.mean() * 100))
print("permutation test, sd         : %.3f" % accuracy_permuted.std())
print("permutation test, 95%% range  : %.1f%% - %.1f%%"
      % (np.percentile(accuracy_permuted, 2.5) * 100, np.percentile(accuracy_permuted, 97.5) * 100))
print("highest value any permutation reached: %.1f%%" % (accuracy_permuted.max() * 100))

# how often was a shuffled run as good as the real one?
p_value = np.mean(accuracy_permuted >= real_acc)
print("\np-value (share of permutations at least as good as the real result): %.4f" % p_value)

# what would simply always saying the bigger class give?
majority = max((realClassLabels == 20).mean(), (realClassLabels == 21).mean())
print("\nalways predicting the bigger class would give: %.1f%%" % (majority * 100))
print("(163 flexion vs 151 extension trials, so chance is not exactly 50%%)")

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

**How is the chance level estimated and what do we use it for?**

It is estimated with a **permutation test**. The idea is to destroy the connection between the data and the labels, and then see how well the classifier still does. In the loop above this is the key line:

```python
y_train_permuted = np.random.permutation(y_train)
```

The training labels are **shuffled randomly**, so a trial keeps its brain data but gets a label that belongs to some other trial. The SVM is then trained on this nonsense and tested on the real test labels. Because there is nothing left to learn, whatever accuracy comes out is pure chance. This is repeated **400 times** with a different shuffle each time, which gives a whole distribution of "accuracies you can get by luck".

We need this because **50% is not the right chance level here**. Our classes are not the same size (163 flexion against 151 extension), so a classifier that always says "flexion" would already get **51.9%** without doing anything. On top of that we only have 314 trials and use cross-validation, so even a useless classifier will sometimes get 55% or 60% just by luck. The permutation test takes all of this into account automatically, because it uses exactly the same data, the same class sizes and the same CV procedure.

Then it is used to judge our real result:

| | accuracy |
| :--- | ---: |
| real SVM | **75.4%** |
| permutation test, mean | 46.6% |
| permutation test, 95% range | 40.3% - 52.8% |
| best value any shuffle reached | 53.8% |

The real accuracy is far outside the range that shuffled labels can produce. Not a single permutation came even close to 75.4%, so the p-value is basically 0. This means the classifier really did find information about the finger movement in the ECoG data, and the 75.4% is not just luck. In the error bar plot this is exactly what you see: the two error bars do not overlap at all.

**Why is this an estimation of the chance level and not the exact value?**

Because it is based on a **limited number of random shuffles**. Every permutation is one random draw, and 400 of them give only a sample of all the possible shuffles, not all of them. If I run the whole thing again with different random seeds, I get a slightly different mean and a slightly different spread. With more repetitions the estimate gets more precise, but it never becomes exact.

There is also no formula that gives the true chance level for this situation, because it depends on the class sizes, on the number of trials, on the CV split and on the classifier itself. So the permutation test is a way of **measuring** it instead of calculating it, which is a Monte Carlo estimate.

One more thing I noticed in the code: `score` and `accuracy_permuted[i] = score` are written **outside** the inner `for k` loop, so only the **last** of the 10 folds is actually used, and the other 9 are thrown away. That still gives an unbiased chance level, but it is based on only about 31 trials instead of all 314, so it is much noisier. When I compared the two versions, the standard deviation was **0.112** the way it is written and only **0.032** when all folds are used, so about 3.5 times wider. The estimate of the chance level is therefore less precise than it could be, but the conclusion does not change, because 75.4% is far above both versions.